In [7]:
# ============================================================
# HVLT FOR TELUGU HANDWRITTEN WORD RECOGNITION
# FULL SINGLE-CELL TRAINING NOTEBOOK
# ============================================================

# ============================================================
# INSTALLS (RUN ONCE IF NEEDED)
# ============================================================

# !pip install timm transformers jiwer opencv-python pillow tqdm scikit-learn

# ============================================================
# IMPORTS
# ============================================================

import os
import cv2
import time
import random
import unicodedata
import numpy as np

from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam

from torch.amp import autocast, GradScaler

import timm

from transformers import XLMRobertaModel

from jiwer import wer

# ============================================================
# CONFIG
# ============================================================

ROOT_DIR = "/home/mca/Shweta/handwritten text detection/dataset/Word_Level_Telugu_Training_Set"

IMG_HEIGHT = 32
IMG_WIDTH = 192

BATCH_SIZE = 32

LR = 5e-5

MAX_EPOCHS = 50

MAX_SEQ_LEN = 40

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

NUM_WORKERS = 4

SEED = 42

USE_AMP = True

# ============================================================
# SEED
# ============================================================

def set_seed(seed=42):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ============================================================
# TELUGU VOCABULARY
# ============================================================

TELUGU_CHARS = (
    "అఆఇఈఉఊఋఎఏఐఒఓఔ"
    "కఖగఘఙచఛజఝఞ"
    "టఠడఢణతథదధన"
    "పఫబభమయరలవ"
    "శషసహళక్షఱ"
    "ాిీుూృెేైొోౌ్ంః"
    "౦౧౨౩౪౫౬౭౮౯"
    "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"
    ".,!?()[]{}:;-_'/\\&%$#@+*= "
)

SPECIAL_TOKENS = [
    "<PAD>",
    "<SOS>",
    "<EOS>",
    "<UNK>",
]

ALL_TOKENS = SPECIAL_TOKENS + list(TELUGU_CHARS)

char2idx = {
    c: i for i, c in enumerate(ALL_TOKENS)
}

idx2char = {
    i: c for c, i in char2idx.items()
}

VOCAB_SIZE = len(ALL_TOKENS)

PAD_IDX = char2idx["<PAD>"]
SOS_IDX = char2idx["<SOS>"]
EOS_IDX = char2idx["<EOS>"]
UNK_IDX = char2idx["<UNK>"]

print("VOCAB SIZE:", VOCAB_SIZE)

# ============================================================
# TELUGU DATASET PATHS
# ============================================================

SET1_DIR = os.path.join(
    ROOT_DIR,
    "Word_Level_Training_Set",
    "Word_Level_Training_Set1"
)

SET2_DIR = os.path.join(
    ROOT_DIR,
    "Word_Level_Training_Set",
    "Word_Level_Training_Set2"
)

TXT1 = os.path.join(
    SET1_DIR,
    "train1.txt"
)

TXT2 = os.path.join(
    SET2_DIR,
    "train2.txt"
)

# ============================================================
# LOAD TELUGU DATA
# ============================================================

samples = []

def load_dataset(txt_path):

    samples = []

    with open(txt_path, "r", encoding="utf-8") as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            parts = line.split("\t")

            if len(parts) != 2:
                continue

            rel_path, text = parts

            text = unicodedata.normalize(
                "NFC",
                text.strip()
            )

            img_path = os.path.join(
                os.path.dirname(txt_path),
                rel_path
            )

            if os.path.exists(img_path):

                samples.append(
                    (img_path, text)
                )

    return samples

samples = []

samples.extend(load_dataset(TXT1))
samples.extend(load_dataset(TXT2))

print("TOTAL SAMPLES:", len(samples))

for i in range(5):
    print(samples[i])

# ============================================================
# SPLIT DATASET
# ============================================================

train_samples, temp_samples = train_test_split(
    samples,
    test_size=0.15,
    random_state=SEED,
)

val_samples, test_samples = train_test_split(
    temp_samples,
    test_size=0.5,
    random_state=SEED,
)

print("TRAIN:", len(train_samples))
print("VAL:", len(val_samples))
print("TEST:", len(test_samples))

# ============================================================
# IMAGE PREPROCESS
# ============================================================

def preprocess_image(img_path):

    img = cv2.imread(img_path)

    if img is None:
        raise ValueError(f"Cannot read image: {img_path}")

    img = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2RGB
    )

    h, w = img.shape[:2]

    if h <= 0 or w <= 0:
        raise ValueError(f"Invalid image size: {img_path}")

    scale = IMG_HEIGHT / h

    new_w = max(1, int(w * scale))

    img = cv2.resize(
        img,
        (new_w, IMG_HEIGHT)
    )

    if new_w < IMG_WIDTH:

        pad = np.ones(
            (
                IMG_HEIGHT,
                IMG_WIDTH - new_w,
                3
            ),
            dtype=np.uint8
        ) * 255

        img = np.concatenate(
            [img, pad],
            axis=1
        )

    else:

        img = cv2.resize(
            img,
            (IMG_WIDTH, IMG_HEIGHT)
        )

    img = img.astype(np.float32) / 255.0

    img = (img - 0.5) / 0.5

    img = np.transpose(
        img,
        (2, 0, 1)
    )

    return torch.tensor(
        img,
        dtype=torch.float32
    )

# ============================================================
# TEXT ENCODING
# ============================================================

def encode_text(text):

    tokens = [SOS_IDX]

    for ch in text:

        tokens.append(
            char2idx.get(ch, UNK_IDX)
        )

    tokens.append(EOS_IDX)

    if len(tokens) < MAX_SEQ_LEN:

        tokens += [PAD_IDX] * (
            MAX_SEQ_LEN - len(tokens)
        )

    else:

        tokens = tokens[:MAX_SEQ_LEN]

        tokens[-1] = EOS_IDX

    return torch.tensor(
        tokens,
        dtype=torch.long
    )

# ============================================================
# TEXT DECODING
# ============================================================

def decode_tokens(tokens):

    chars = []

    for t in tokens:

        t = int(t)

        if t == EOS_IDX:
            break

        if t in [PAD_IDX, SOS_IDX]:
            continue

        chars.append(
            idx2char.get(t, "")
        )

    return "".join(chars)

# ============================================================
# DATASET
# ============================================================

class TeluguWordDataset(Dataset):

    def __init__(self, samples):

        self.samples = samples

    def __len__(self):

        return len(self.samples)

    def __getitem__(self, idx):

        img_path, text = self.samples[idx]

        image = preprocess_image(img_path)

        tokens = encode_text(text)

        return image, tokens, text

# ============================================================
# DATALOADERS
# ============================================================

train_loader = DataLoader(
    TeluguWordDataset(train_samples),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

val_loader = DataLoader(
    TeluguWordDataset(val_samples),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

test_loader = DataLoader(
    TeluguWordDataset(test_samples),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

# ============================================================
# POSITIONAL BRIDGE
# ============================================================

class PositionalBridge(nn.Module):

    def __init__(
        self,
        in_channels=1024,
        d_model=768,
        vis_seq_len=256,
    ):
        super().__init__()

        self.pool = nn.AdaptiveAvgPool2d(
            (1, vis_seq_len)
        )

        self.proj = nn.Linear(
            in_channels,
            d_model,
        )

        self.pos_embed = nn.Parameter(
            torch.randn(
                1,
                vis_seq_len,
                d_model,
            ) * 0.02
        )

    def forward(self, x):

        B, H, W, C = x.shape

        x = x.permute(0, 3, 1, 2)

        x = self.pool(x)

        x = x.squeeze(2)

        x = x.permute(0, 2, 1)

        x = self.proj(x)

        x = x + self.pos_embed

        return x

# ============================================================
# DECODER
# ============================================================

class TeluguDecoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model=768,
        n_heads=12,
        n_layers=6,
    ):
        super().__init__()

        self.token_embed = nn.Embedding(
            vocab_size,
            d_model,
            padding_idx=PAD_IDX,
        )

        self.pos_embed = nn.Embedding(
            MAX_SEQ_LEN,
            d_model,
        )

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=3072,
            dropout=0.1,
            batch_first=True,
        )

        self.decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=n_layers,
        )

        self.output_proj = nn.Linear(
            d_model,
            vocab_size,
        )

        print("Loading XLM-RoBERTa...")

        try:

            xlm = XLMRobertaModel.from_pretrained(
                "xlm-roberta-base"
            )

            emb = xlm.embeddings.word_embeddings.weight

            n = min(
                emb.shape[0],
                vocab_size,
            )

            self.token_embed.weight.data[:n] = emb[:n]

            del xlm

            print("Loaded multilingual weights.")

        except Exception as e:

            print("Pretrained load failed:", e)

    def forward(
        self,
        memory,
        tgt_tokens,
    ):

        B, T = tgt_tokens.shape

        pos = torch.arange(
            T,
            device=tgt_tokens.device,
        ).unsqueeze(0).expand(B, -1)

        x = (
            self.token_embed(tgt_tokens)
            + self.pos_embed(pos)
        )

        tgt_mask = torch.triu(
            torch.ones(
                T,
                T,
                device=tgt_tokens.device,
            ),
            diagonal=1,
        ).bool()

        tgt_key_padding_mask = (
            tgt_tokens == PAD_IDX
        )

        out = self.decoder(
            tgt=x,
            memory=memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
        )

        logits = self.output_proj(out)

        return logits

    @torch.no_grad()
    def greedy_decode(
        self,
        memory,
        max_len=MAX_SEQ_LEN,
    ):

        B = memory.shape[0]

        generated = torch.full(
            (B, 1),
            SOS_IDX,
            device=memory.device,
            dtype=torch.long,
        )

        finished = torch.zeros(
            B,
            dtype=torch.bool,
            device=memory.device,
        )

        for _ in range(max_len):

            logits = self.forward(
                memory,
                generated,
            )

            next_token = logits[:, -1].argmax(dim=-1)

            next_token = next_token.masked_fill(
                finished,
                PAD_IDX,
            )

            generated = torch.cat(
                [
                    generated,
                    next_token.unsqueeze(1),
                ],
                dim=1,
            )

            finished |= (
                next_token == EOS_IDX
            )

            if finished.all():
                break

        return generated[:, 1:]

# ============================================================
# HVLT MODEL
# ============================================================

class HVLT(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=True,
            features_only=True,
            img_size=(32, 192),
            strict_img_size=False,
        )

        self.bridge = PositionalBridge(
            in_channels=1024,
            d_model=768,
            vis_seq_len=256,
        )

        self.decoder = TeluguDecoder(
            vocab_size=VOCAB_SIZE
        )

    def forward(
        self,
        images,
        tgt_tokens,
    ):

        feats = self.encoder(images)[-1]

        memory = self.bridge(feats)

        logits = self.decoder(
            memory,
            tgt_tokens,
        )

        return logits

    @torch.no_grad()
    def predict(self, images):

        feats = self.encoder(images)[-1]

        memory = self.bridge(feats)

        preds = self.decoder.greedy_decode(memory)

        return preds

# ============================================================
# LOSS
# ============================================================

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_IDX
)

# ============================================================
# METRICS
# ============================================================

def char_accuracy(preds, labels):

    correct = 0

    total = 0

    for p, l in zip(preds, labels):

        n = min(len(p), len(l))

        for i in range(n):

            if p[i] == l[i]:
                correct += 1

        total += max(len(p), len(l))

    return 100.0 * correct / max(total, 1)

# ============================================================
# MODEL
# ============================================================

model = HVLT().to(DEVICE)

optimizer = Adam(
    model.parameters(),
    lr=LR,
)

scaler = GradScaler(
    "cuda",
    enabled=USE_AMP
)

print(
    "TOTAL PARAMS:",
    sum(
        p.numel()
        for p in model.parameters()
    ) / 1e6,
    "M"
)

# ============================================================
# TRAINING LOOP
# ============================================================

best_wer = 9999

for epoch in range(1, MAX_EPOCHS + 1):

    print("\n")
    print("=" * 70)
    print(f"STARTING EPOCH {epoch}/{MAX_EPOCHS}")
    print("=" * 70)

    model.train()

    train_loss = 0

    train_preds = []

    train_labels = []

    pbar = tqdm(
        train_loader,
        desc=f"Epoch {epoch}/{MAX_EPOCHS}"
    )

    for images, targets, labels in pbar:

        images = images.to(DEVICE)

        targets = targets.to(DEVICE)

        decoder_input = targets[:, :-1]

        decoder_target = targets[:, 1:]

        optimizer.zero_grad()

        with autocast(
            "cuda",
            enabled=USE_AMP
        ):

            logits = model(
                images,
                decoder_input,
            )

            loss = criterion(
                logits.reshape(
                    -1,
                    VOCAB_SIZE
                ),
                decoder_target.reshape(-1),
            )

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            5.0,
        )

        scaler.step(optimizer)

        scaler.update()

        train_loss += loss.item()

        with torch.no_grad():

            pred_ids = logits.argmax(dim=-1)

            preds = [
                decode_tokens(x)
                for x in pred_ids
            ]

        train_preds.extend(preds)

        train_labels.extend(labels)

        pbar.set_postfix(
            loss=f"{loss.item():.4f}",
            epoch=f"{epoch}/{MAX_EPOCHS}"
        )

    train_cer = char_accuracy(
        train_preds,
        train_labels,
    )

    train_wer = wer(
        train_labels,
        train_preds,
    ) * 100

    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    val_loss = 0

    val_preds = []

    val_labels = []

    with torch.no_grad():

        pbar = tqdm(
            val_loader,
            desc=f"Validation {epoch}"
        )

        for images, targets, labels in pbar:

            images = images.to(DEVICE)

            targets = targets.to(DEVICE)

            decoder_input = targets[:, :-1]

            decoder_target = targets[:, 1:]

            logits = model(
                images,
                decoder_input,
            )

            loss = criterion(
                logits.reshape(
                    -1,
                    VOCAB_SIZE
                ),
                decoder_target.reshape(-1),
            )

            val_loss += loss.item()

            pred_ids = model.predict(images)

            preds = [
                decode_tokens(x)
                for x in pred_ids
            ]

            val_preds.extend(preds)

            val_labels.extend(labels)

    val_cer = char_accuracy(
        val_preds,
        val_labels,
    )

    val_wer = wer(
        val_labels,
        val_preds,
    ) * 100

    print("\n")
    print("=" * 70)

    print(f"EPOCH {epoch} COMPLETED")

    print(
        f"TRAIN LOSS: {train_loss / len(train_loader):.4f}"
    )

    print(
        f"VAL LOSS: {val_loss / len(val_loader):.4f}"
    )

    print(
        f"TRAIN CAR: {train_cer:.2f}%"
    )

    print(
        f"VAL CAR: {val_cer:.2f}%"
    )

    print(
        f"TRAIN WER: {train_wer:.2f}%"
    )

    print(
        f"VAL WER: {val_wer:.2f}%"
    )

    print("=" * 70)

    if val_wer < best_wer:

        best_wer = val_wer

        torch.save(
            {
                "model": model.state_dict(),
                "epoch": epoch,
                "val_wer": val_wer,
            },
            "best_telugu_hvlt.pth",
        )

        print("BEST MODEL SAVED.")

# ============================================================
# TEST EVALUATION
# ============================================================

print("\nFINAL TEST EVALUATION")

model.eval()

test_preds = []

test_labels = []

with torch.no_grad():

    for images, targets, labels in tqdm(test_loader):

        images = images.to(DEVICE)

        pred_ids = model.predict(images)

        preds = [
            decode_tokens(x)
            for x in pred_ids
        ]

        test_preds.extend(preds)

        test_labels.extend(labels)

test_cer = char_accuracy(
    test_preds,
    test_labels,
)

test_wer = wer(
    test_labels,
    test_preds,
) * 100

print("\nTEST RESULTS")

print("TEST CAR:", test_cer)

print("TEST WER:", test_wer)

# ============================================================
# SAMPLE PREDICTIONS
# ============================================================

print("\nSAMPLE PREDICTIONS\n")

for i in range(10):

    print("GT   :", test_labels[i])

    print("PRED :", test_preds[i])

    print("-" * 50)

VOCAB SIZE: 158
TOTAL SAMPLES: 77625
('/home/mca/Shweta/handwritten text detection/dataset/Word_Level_Telugu_Training_Set/Word_Level_Training_Set/Word_Level_Training_Set1/image1/45377_1.jpg', 'అది')
('/home/mca/Shweta/handwritten text detection/dataset/Word_Level_Telugu_Training_Set/Word_Level_Training_Set/Word_Level_Training_Set1/image1/45377_2.jpg', 'వెలువడకపోయినా')
('/home/mca/Shweta/handwritten text detection/dataset/Word_Level_Telugu_Training_Set/Word_Level_Training_Set/Word_Level_Training_Set1/image1/45377_3.jpg', 'దానిలోని')
('/home/mca/Shweta/handwritten text detection/dataset/Word_Level_Telugu_Training_Set/Word_Level_Training_Set/Word_Level_Training_Set1/image1/45377_4.jpg', 'పరిశోధకమండలి')
('/home/mca/Shweta/handwritten text detection/dataset/Word_Level_Telugu_Training_Set/Word_Level_Training_Set/Word_Level_Training_Set1/image1/45377_5.jpg', 'పత్రికలో')
TRAIN: 65981
VAL: 5822
TEST: 5822
Loading XLM-RoBERTa...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded multilingual weights.
TOTAL PARAMS: 144.660406 M


STARTING EPOCH 1/50


Validation 1: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 182/182 [00:37<00:00,  4.84it/s]


...



EPOCH 49 COMPLETED
TRAIN LOSS: 0.0173
VAL LOSS: 0.3375
TRAIN CAR: 87.67%
VAL CAR: 73.17%
TRAIN WER: 6.80%
VAL WER: 28.55%


STARTING EPOCH 50/50


Validation 50: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 182/182 [00:35<00:00,  5.18it/s]




EPOCH 50 COMPLETED
TRAIN LOSS: 0.0159
VAL LOSS: 0.3413
TRAIN CAR: 87.71%
VAL CAR: 72.62%
TRAIN WER: 6.55%
VAL WER: 28.99%

FINAL TEST EVALUATION


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 182/182 [00:30<00:00,  6.06it/s]


TEST RESULTS
TEST CAR: 72.17410801872249
TEST WER: 29.2167639986259

SAMPLE PREDICTIONS

GT   : ఈ
PRED : ఈ
--------------------------------------------------
GT   : చేరాడు
PRED : చేశాడు
--------------------------------------------------
GT   : తరువాత
PRED : తరువాత
--------------------------------------------------
GT   : ,
PRED : ,
--------------------------------------------------
GT   : నిరంతరంగా
PRED : నిరంతరంగా
--------------------------------------------------
GT   : సమాలు
PRED : సమాలు
--------------------------------------------------
GT   : ఉండి
PRED : ఉం
--------------------------------------------------
GT   : గుర్తిస్తారు
PRED : గుర్తిస్తారు
--------------------------------------------------
GT   : ఉంచబడింది
PRED : ఉంచబడింది
--------------------------------------------------
GT   : అర్ధాలు
PRED : అర్దాలు
--------------------------------------------------


In [2]:
import os

ROOT_DIR = "/home/mca/Shweta/handwritten text detection/dataset/Word_Level_Telugu_Training_Set"

for root, dirs, files in os.walk(ROOT_DIR):
    if "train1.txt" in files:
        print("FOUND TRAIN1:", os.path.join(root, "train1.txt"))

    if "train2.txt" in files:
        print("FOUND TRAIN2:", os.path.join(root, "train2.txt"))

FOUND TRAIN2: /home/mca/Shweta/handwritten text detection/dataset/Word_Level_Telugu_Training_Set/Word_Level_Training_Set/Word_Level_Training_Set2/train2.txt
FOUND TRAIN1: /home/mca/Shweta/handwritten text detection/dataset/Word_Level_Telugu_Training_Set/Word_Level_Training_Set/Word_Level_Training_Set1/train1.txt


In [3]:
with open(TXT1, "r", encoding="utf-8") as f:
    for i in range(5):
        print(repr(next(f)))

'image1/45377_1.jpg\tఅది\n'
'image1/45377_2.jpg\tవెలువడకపోయినా\n'
'image1/45377_3.jpg\tదానిలోని\n'
'image1/45377_4.jpg\tపరిశోధకమండలి\n'
'image1/45377_5.jpg\tపత్రికలో\n'


In [4]:
import os

txt_dir = "/home/mca/Shweta/handwritten text detection/dataset/Word_Level_Telugu_Training_Set/Word_Level_Training_Set/Word_Level_Training_Set1"

img_path = os.path.join(
    txt_dir,
    "image1/45377_1.jpg"
)

print(img_path)
print(os.path.exists(img_path))

/home/mca/Shweta/handwritten text detection/dataset/Word_Level_Telugu_Training_Set/Word_Level_Training_Set/Word_Level_Training_Set1/image1/45377_1.jpg
True
